# Experimento 4: Precisión del Reconocedor en Condiciones Controladas

## Objetivo
Medir la precisión del reconocedor de rostros y evaluar su robustez bajo cuatro condiciones específicas. Esto nos permitirá calcular el FAR (False Acceptance Rate) y FRR (False Rejection Rate) de forma estructurada.

## Condiciones a evaluar
Asegúrate de colocar al menos 5 imágenes en cada una de las siguientes carpetas dentro de `dataset/exp4_conditions/`:

- **Condición A (`condicion_A_frontal`)**: Rostros frontales con buena iluminación. (Subcarpetas por persona)
- **Condición B (`condicion_B_perfil`)**: Rostros de perfil, ángulos difíciles o accesorios. (Subcarpetas por persona)
- **Condición C (`condicion_C_desconocidos`)**: Personas no registradas en la base de datos.
- **Condición D (`condicion_D_multiples`)**: Dos o más personas registradas en la misma foto. **Nota**: El nombre del archivo debe contener los primeros nombres de quienes aparecen (ej. `gerardo_kevin_01.jpg`) para poder evaluarlo automáticamente.

## Criterio de Aceptación
Generar tablas por condición, gráficos comparativos, FAR, FRR e interpretar los resultados. Todo debe guardarse en `experiments/resultados_exp4.csv`.

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from ultralytics import YOLO

# Raíz del proyecto
RAIZ = Path.cwd()
if RAIZ.name == "experiments":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.recognizer import FaceRecognizer
from src.configuracion import YOLO_WEIGHTS, EMBEDDINGS_PATH, RECOGNITION_THRESHOLD

# Inicializar modelos
modelo_yolo = YOLO(str(YOLO_WEIGHTS))
recognizer = FaceRecognizer(db_path=str(EMBEDDINGS_PATH), threshold=RECOGNITION_THRESHOLD)

PERSONAS_REGISTRADAS = list(recognizer.embeddings_db.keys())
print(f"Personas en DB: {PERSONAS_REGISTRADAS}")

In [ ]:
def detectar_y_recortar(imagen, conf=0.5, margen=0.3):
    """Detecta y recorta todos los rostros en una imagen."""
    results = modelo_yolo.predict(imagen, conf=conf, verbose=False)
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return []
    
    boxes = results[0].boxes.xyxy.cpu().numpy()
    scores = results[0].boxes.conf.cpu().numpy()
    
    rostros = []
    h, w = imagen.shape[:2]
    for b, s in zip(boxes, scores):
        x1, y1, x2, y2 = int(b[0]), int(b[1]), int(b[2]), int(b[3])
        mx, my = int((x2 - x1) * margen), int((y2 - y1) * margen)
        rx1, ry1 = max(0, x1 - mx), max(0, y1 - my)
        rx2, ry2 = min(w, x2 + mx), min(h, y2 + my)
        
        if ry2 > ry1 and rx2 > rx1:
            rostros.append({
                "crop": imagen[ry1:ry2, rx1:rx2],
                "conf": float(s)
            })
    return rostros


In [ ]:
# Ejecutar pipeline por condiciones
BASE_DIR = RAIZ / "dataset" / "exp4_conditions"
resultados = []

condiciones = [
    ("A_Frontal", BASE_DIR / "condicion_A_frontal"),
    ("B_Perfil", BASE_DIR / "condicion_B_perfil"),
    ("C_Desconocidos", BASE_DIR / "condicion_C_desconocidos"),
    ("D_Multiples", BASE_DIR / "condicion_D_multiples")
]

for nombre_cond, ruta_cond in condiciones:
    if not ruta_cond.exists():
        print(f"⚠️ Falta carpeta: {ruta_cond}")
        continue
        
    if nombre_cond in ["A_Frontal", "B_Perfil"]:
        # Carpetas por persona
        for carpeta_persona in ruta_cond.iterdir():
            if not carpeta_persona.is_dir(): continue
            esperado = carpeta_persona.name
            for foto in carpeta_persona.glob("*.*"):
                if foto.suffix.lower() not in [".jpg", ".jpeg", ".png"]: continue
                
                img = cv2.imread(str(foto))
                if img is None: continue
                rostros = detectar_y_recortar(img)
                
                if not rostros:
                    resultados.append({"condicion": nombre_cond, "archivo": foto.name, "esperado": esperado, "devuelto": "Sin detección", "distancia": 0, "tipo_error": "FN"})
                    continue
                
                # Tomar el de mayor confianza
                mejor_rostro = max(rostros, key=lambda r: r["conf"])
                devuelto, dist = recognizer.recognize(mejor_rostro["crop"])
                
                tipo = "Correcto"
                if devuelto != esperado:
                    tipo = "FN" if devuelto == "Desconocido" else "Confusion"
                    
                resultados.append({"condicion": nombre_cond, "archivo": foto.name, "esperado": esperado, "devuelto": devuelto, "distancia": dist, "tipo_error": tipo})
                
    elif nombre_cond == "C_Desconocidos":
        # Esperado = Desconocido
        for foto in ruta_cond.glob("*.*"):
            if foto.suffix.lower() not in [".jpg", ".jpeg", ".png"]: continue
            img = cv2.imread(str(foto))
            if img is None: continue
            rostros = detectar_y_recortar(img)
            
            if not rostros:
                resultados.append({"condicion": nombre_cond, "archivo": foto.name, "esperado": "Desconocido", "devuelto": "Sin detección", "distancia": 0, "tipo_error": "Correcto"})
                continue
                
            mejor_rostro = max(rostros, key=lambda r: r["conf"])
            devuelto, dist = recognizer.recognize(mejor_rostro["crop"])
            
            tipo = "Correcto" if devuelto == "Desconocido" else "FP"
            resultados.append({"condicion": nombre_cond, "archivo": foto.name, "esperado": "Desconocido", "devuelto": devuelto, "distancia": dist, "tipo_error": tipo})
            
    elif nombre_cond == "D_Multiples":
        for foto in ruta_cond.glob("*.*"):
            if foto.suffix.lower() not in [".jpg", ".jpeg", ".png"]: continue
            img = cv2.imread(str(foto))
            if img is None: continue
            
            # Inferir esperados por nombre de archivo
            esperados = [p for p in PERSONAS_REGISTRADAS if p.split("_")[0].lower() in foto.name.lower()]
            
            rostros = detectar_y_recortar(img)
            
            nombres_devueltos = []
            for r in rostros:
                devuelto, dist = recognizer.recognize(r["crop"])
                nombres_devueltos.append((devuelto, dist))
                
            # Evaluar a nivel de frame completo
            devueltos_conocidos = [n for n, d in nombres_devueltos if n != "Desconocido"]
            
            fps = [n for n in devueltos_conocidos if n not in esperados]
            fns = [e for e in esperados if e not in devueltos_conocidos]
            
            tipo = "Correcto"
            if fps: tipo = "FP"
            if fns: tipo = "FN"
            if fps and fns: tipo = "FP+FN"
            
            resultados.append({"condicion": nombre_cond, "archivo": foto.name, "esperado": ",".join(esperados), "devuelto": ",".join(devueltos_conocidos), "distancia": 0, "tipo_error": tipo})

print(f"Se procesaron {len(resultados)} imágenes/frames.")


## Resultados por Condición

In [ ]:
df = pd.DataFrame(resultados)
if not df.empty:
    df.to_csv(RAIZ / "experiments" / "resultados_exp4.csv", index=False)
    
    print("--- DESGLOSE POR TIPO DE ERROR ---")
    display(df.groupby(["condicion", "tipo_error"]).size().unstack(fill_value=0))
    
    # Accuracy por condicion
    acc_por_condicion = []
    for cond in df["condicion"].unique():
        sub = df[df["condicion"] == cond]
        corr = len(sub[sub["tipo_error"] == "Correcto"])
        acc_por_condicion.append({"Condicion": cond, "Accuracy": corr / len(sub) * 100})
        
    df_acc = pd.DataFrame(acc_por_condicion)
    print("\n--- ACCURACY POR CONDICIÓN ---")
    display(df_acc.style.format({"Accuracy": "{:.1f}%"}))
    
    # Grafico de barras
    plt.figure(figsize=(8, 5))
    plt.bar(df_acc["Condicion"], df_acc["Accuracy"], color="#3498db")
    plt.title("Accuracy por Condición (Experimento 4)")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 110)
    for i, v in enumerate(df_acc["Accuracy"]):
        plt.text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(RAIZ / "experiments" / "graphs" / "exp4_accuracy_condiciones.png")
    plt.show()
else:
    print("No hay resultados. Asegúrate de colocar imágenes en dataset/exp4_conditions/")


## Cálculo de FAR y FRR Globales

- **FAR (False Acceptance Rate)**: Falsos Positivos / (Falsos Positivos + Verdaderos Negativos). Es la tasa a la que un desconocido es aceptado por el sistema como si estuviera registrado.
- **FRR (False Rejection Rate)**: Falsos Negativos / (Falsos Negativos + Verdaderos Positivos). Es la tasa a la que un usuario registrado es rechazado erróneamente por el sistema.

In [ ]:
if not df.empty:
    FP = len(df[df["tipo_error"].isin(["FP", "FP+FN", "Confusion"])])
    FN = len(df[df["tipo_error"].isin(["FN", "FP+FN", "Confusion"])])
    
    # Estimación de VN y VP 
    VN = len(df[(df["esperado"] == "Desconocido") & (df["tipo_error"] == "Correcto")])
    VP = len(df[(df["esperado"] != "Desconocido") & (df["tipo_error"] == "Correcto")])
    
    FAR = FP / (FP + VN) if (FP + VN) > 0 else 0
    FRR = FN / (FN + VP) if (FN + VP) > 0 else 0
    
    print("========================================")
    print(f"FAR Global (Riesgo de Seguridad):  {FAR*100:.2f}%")
    print(f"FRR Global (Fricción / Comodidad): {FRR*100:.2f}%")
    print("========================================")


## Interpretación de los Resultados

*(A completar luego de ejecutar con los datos reales)*

### Condición A (Frontal - Caso Ideal)
Se espera un accuracy cercano al 100%. Si falla aquí, significa que la base de embeddings tiene problemas de calidad o el umbral es extremadamente estricto.

### Condición B (Perfil y Accesorios - Caso Difícil)
Es normal que el accuracy baje, aumentando el FRR. Esto nos indica qué tanto puede tolerar el sistema antes de fallar.

### Condición C (Desconocidos - Seguridad)
Debe tener un 100% de Verdaderos Negativos (Accuracy 100% en esta categoría). Un fallo aquí significa un intruso aceptado (incrementa el FAR), lo cual es crítico.

### Condición D (Múltiples rostros)
Evalúa si el reconocedor y el detector pueden manejar inferencia concurrente sin confundir IDs.